In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import xgboost as xgb
import shap
from sklearn.linear_model import QuantileRegressor
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

train = pd.read_csv("../data/train.csv")
val   = pd.read_csv("../data/val.csv")
test  = pd.read_csv("../data/test.csv")

FEATURE_COLS = joblib.load("../model/feature_cols.pkl")
TARGET_COL   = "total_delivery_min"

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Train: 6517 | Val: 1769 | Test: 1714


In [2]:
def evaluate(y_true, y_low, y_high, label=""):
    within = ((y_true >= y_low) & (y_true <= y_high)).mean()
    late   = (y_true > y_high).mean()
    early  = (y_true < y_low).mean()
    width  = (y_high - y_low).mean()

    print(f"{label}")
    print(f"  Coverage rate:   {within*100:.1f}%")
    print(f"  Late rate:       {late*100:.1f}%   (llegaron después del promise_end, OBJETIVO 3%)")
    print(f"  Early rate:      {early*100:.1f}%  (llegaron antes del promise_start)")
    print(f"  Avg window:      {width:.1f} min")

    return {"coverage": within, "late": late, "early": early, "width": width}

In [ ]:
TAU_LOW  = 0.15
TAU_HIGH = 0.97 # 3% llega tarde

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

lr_low  = QuantileRegressor(quantile=TAU_LOW,  alpha=0.1, solver="highs")
lr_high = QuantileRegressor(quantile=TAU_HIGH, alpha=0.1, solver="highs")

lr_low.fit(X_train_sc,  y_train)
lr_high.fit(X_train_sc, y_train)

val_low_lr  = lr_low.predict(X_val_sc)
val_high_lr = lr_high.predict(X_val_sc)

metrics_lr_val = evaluate(y_val, val_low_lr, val_high_lr, "Lineal — Validación")